# ਪਾਠ 10 - ਉਤਪਾਦਨ ਵਿੱਚ ਏਆਈ ਏਜੰਟ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਮਾਈਕਰੋਸਾਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ (ਪਾਇਥਨ) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਏਆਈ ਏਜੰਟਾਂ ਲਈ **ਉਤਪਾਦਨ ਪੈਟਰਨ** ਸਿਖੋਗੇ। ਅਸੀਂ ਕਵਰ ਕਰਦੇ ਹਾਂ:

- **ਪ੍ਰਵੇਸਯੋਗਤਾ** — ਏਜੰਟ ਪਰਸਪਰਕ੍ਰਿਆਵਾਂ ਵਿੱਚ ਸਮਾਂ ਨਿਰਧਾਰਿਤ ਕਰਨ ਅਤੇ ਲਾਗਿੰਗ ਸ਼ਾਮਲ ਕਰਨਾ
- **ਮੁੱਲਾਂਕਣ** — ਜਵਾਬ ਦੀ ਗੁਣਵੱਤਾ ਨੂੰ ਸਕੋਰ ਕਰਨ ਲਈ ਇੱਕ ਮੁੱਲਾਂਕਣ ਏਜੰਟ ਦੀ ਵਰਤੋਂ
- **ਲਾਗਤ ਪ੍ਰਬੰਧਨ** — ਟੋਕਨ ਅਪਟਿਮਾਈਜ਼ੇਸ਼ਨ ਅਤੇ ਮਾਡਲ ਚੋਣ ਲਈ ਰਣਨੀਤੀਆਂ

ਪਰਿਦ੍ਰਿਸ਼ ਇੱਕ **ਯਾਤਰਾ ਏਜੰਟ** ਹੈ ਜੋ ਉਪਭੋਗਤਾਵਾਂ ਦੀ ਯਾਤਰਾ ਯੋਜਨਾ ਬਣਾਉਣ ਵਿੱਚ ਮਦਦ ਕਰਦਾ ਹੈ, ਜਿਸ 'ਤੇ ਨਿਗਰਾਨੀ ਅਤੇ ਮੁੱਲਾਂਕਣ ਦੀ ਪਰਤ ਲਾਈ ਗਈ ਹੈ।


## ਸੈਟਅੱਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ਉਤਪਾਦਨ ਵਿਚਾਰ

ਪ੍ਰੋਟੋਟਾਈਪ ਤੋਂ ਉਤਪਾਦਨ ਵੱਲ ਏਆਈ ਏਜੇੰਟਸ ਨੂੰ ਲਿਜਾਣ ਲਈ ਤਿੰਨ ਮੁੱਖ ਅੰਗਾਂ 'ਤੇ ਧਿਆਨ ਦੇਣਾ ਲਾਜ਼ਮੀ ਹੈ:

1. **ਨਿਰੀਖਣਯੋਗਤਾ** — ਤੁਹਾਨੂੰ ਇਹ ਦੇਖਣ ਦੀ ਲੋੜ ਹੈ ਕਿ ਏਜੇੰਟ ਕੀ ਕਰ ਰਿਹਾ ਹੈ, ਇਹ ਕਿੰਨਾ ਸਮਾਂ ਲੈਂਦਾ ਹੈ, ਅਤੇ ਕਿਹੜੇ ਟੂਲ ਇਹ ਕਾਲ ਕਰਦਾ ਹੈ। ਟਰੇਸਿੰਗ ਅਤੇ ਲੌਗਿੰਗ ਦੇ ਬਿਨਾ, ਉਤਪਾਦਨ ਸਮੱਸਿਆਵਾਂ ਦਾ ਡੀਬੱਗ ਕਰਨਾ ਲਗਭਗ ਅਸੰਭਵ ਹੈ।

2. **ਮੁਲਾਂਕਣ** — ਸਵੈਚਾਲਿਤ ਗੁਣਵੱਤਾ ਜਾਂਚਾਂ ਇਹ ਯਕੀਨੀ ਬਣਾਉਂਦੀਆਂ ਹਨ ਕਿ ਏਜੇੰਟ ਦੇ ਜਵਾਬ ਸਮੇਂ ਦੇ ਨਾਲ ਸਹੀ, ਪੂਰੇ ਅਤੇ ਸਹਾਇਕ ਰਹਿਣ। ਇੱਕ ਮੁਲਾਂਕਣ ਏਜੇੰਟ ਜਵਾਬਾਂ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਮਾਪਦੰਡਾਂ ਅਨੁਸਾਰ ਸਕੋਰ ਕਰ ਸਕਦਾ ਹੈ।

3. **ਲਾਗਤ ਪ੍ਰਬੰਧਨ** — ਟੋਕਨ ਦੀ ਵਰਤੋਂ ਸਿੱਧਾ ਲਾਗਤ ਨੂੰ ਪ੍ਰਭਾਵਿਤ ਕਰਦੀ ਹੈ। ਪ੍ਰਾਪਤ ਵਿਧੀਆਂ ਜਿਵੇਂ ਪ੍ਰੰਪਟ ਅਪਟੀਮਾਈਜੇਸ਼ਨ, ਮਾਡਲ ਚੋਣ ਅਤੇ ਕੈਸ਼ਿੰਗ ਖਰਚਿਆਂ ਨੂੰ ਕੁੱਟ-ਕਮ ਕਰਨ ਵਿਚ ਮਦਦ ਕਰਦੀਆਂ ਹਨ ਬਿਨਾ ਗੁਣਵੱਤਾ ਨੂੰ ਨੁਕਸਾਨ ਪੁਚਾਏ।


## ਇੱਕ ਦ੍ਰਿਸ਼ਮਾਨ ਏਜੰਟ ਬਣਾਉਣਾ

ਅਸੀਂ سفر ਦੇ ਸਾਧਨ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਾਂ ਅਤੇ ਏਜੰਟ ਕਾਲ ਨੂੰ ਟਾਈਮਿੰਗ ਨਾਲ ਲਪੇਟਦੇ ਹਾਂ ਤਾਂ ਜੋ ਅਸੀਂ ਲੇਟੈਂਸੀ ਦੀ ਨਿਗਰਾਨੀ ਕਰ ਸਕੀਏ। ਪ੍ਰੋਡਕਸ਼ਨ ਵਿੱਚ ਤੁਸੀਂ OpenTelemetry ਜਾਂ ਇਸੇ ਤਰ੍ਹਾਂ ਦੇ ट्रेसਿੰਗ ਬੈਕਐੰਡ ਨਾਲ ਇੰਟੀਗ੍ਰੇਟ ਕਰੋਗੇ।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## ਮੁਲਾਂਕਣ ਪੈਟਰਨ

ਇੱਕ ਆਮ ਪ੍ਰੋਡਕਸ਼ਨ ਪੈਟਰਨ ਦੂਜੇ ਏਜੰਟ ਨੂੰ **ਮੁਲਾਂਕਣਕਰਤਾ** ਵਜੋਂ ਵਰਤਣਾ ਹੈ। ਮੁਲਾਂਕਣਕਰਤਾ ਮੁੱਖ ਏਜੰਟ ਦੇ ਜਵਾਬ ਨੂੰ ਪੂਰਨਤਾ, ਸਹੀਤਾ ਅਤੇ ਸਹਾਇਤਾ ਜਿਹੇ ਪਹਿਲੂਆਂ ਦੇ ਅਧਾਰ 'ਤੇ ਪ੍ਰੀ-ਤੈਅ ਕੀਤੇ ਮਾਪਦੰਡਾਂ ਨਾਲ ਸਕੋਰ ਕਰਦਾ ਹੈ।

ਇਹ ਯੋਗ ਬਣਾਉਂਦਾ ਹੈ:
- ਉਪਭੋਗਤਾਵਾਂ ਤੱਕ ਜਵਾਬ ਪਹੁੰਚਣ ਤੋਂ ਪਹਿਲਾਂ ਸਵੈਚਾਲਿਤ ਗੁਣਵੱਤਾ ਬੰਦ ਬਟਨ
- ਜਦੋਂ ਪ੍ਰਾਂਪਟ ਜਾਂ ਮਾਡਲ ਬਦਲਦੇ ਹਨ ਤਾਂ ਰਿਗਰੈਸ਼ਨ ਦਾ ਪਤਾ ਲਗਾਉਣਾ
- ਸਮੇਂ ਦੇ ਨਾਲ ਏਜੰਟ ਦੇ ਪ੍ਰਦਰਸ਼ਨ ਦੀ ਲਗਾਤਾਰ ਨਿਗਰਾਨੀ


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ਲਾਗਤ ਪ੍ਰਬੰਧਨ ਰਣਨੀਤੀਆਂ

ਖਰਚਾਂ ਨੂੰ ਕੰਟਰੋਲ ਕਰਨਾ ਉਤਪਾਦਨ ਏਆਈ ਏਜੰਟਾਂ ਲਈ ਅਹਮ ਹੈ। ਇੱਥੇ ਮੁੱਖ ਰਣਨੀਤੀਆਂ ਹਨ:

| ਰਣਨੀਤੀ | ਵਰਣਨ |
|---|---|
| **ਪ੍ਰਾਂਪਟ ਅਪਟਿਮਾਈਜ਼ੇਸ਼ਨ** | ਸਿਸਟਮ ਦੇ ਨਿਰਦੇਸ਼ਾਂ ਨੂੰ ਸੰਖੇਪ ਰੱਖੋ। ਇਨਪੁੱਟ ਟੋਕਨਜ਼ ਘਟਾਉਣ ਲਈ ਵਰਤੋਂ ਵਿੱਚ ਨਾ ਆਉਣ ਵਾਲੇ ਸੰਦਰਭ ਨੂੰ ਹਟਾਓ। |
    "| **ਮਾਡਲ ਚੋਣ** | ਸਧਾਰਨ ਕੰਮਾਂ ਜਿਵੇਂ ਕਿ ਵਰਗੀਕਰਨ ਜਾਂ ਨਿਕਾਸ ਲਈ ਛੋਟੇ, ਸਸਤੇ ਮਾਡਲਾਂ (ਜਿਵੇਂ GPT-4.1-mini) ਦੀ ਵਰਤੋਂ ਕਰੋ, ਅਤੇ ਜਟਿਲ ਤਕਨੀਕੀ ਕਾਰਜਾਂ ਲਈ ਵੱਡੇ ਮਾਡਲਾਂ ਨੂੰ ਰਾਖਵਾਲ ਕਰੋ। |\n",
| **ਕੈਸ਼ਿੰਗ** | ਦੋਹਰਾਏ ਜਾਣ ਵਾਲੇ ਏਪੀਆਈ ਕਾਲਾਂ ਤੋਂ ਬਚਣ ਲਈ ਟੂਲ ਨਤੀਜੇ ਅਤੇ ਅਕਸਰ ਪੁੱਛੇ ਜਾਣ ਵਾਲੇ ਪ੍ਰਸ਼ਨਾਂ ਨੂੰ ਕੈਸ਼ ਕਰੋ। |
| **ਟੋਕਨ ਬਜਟ** | ਅਣਉਮੀਦਤ ਤੌਰ 'ਤੇ ਲੰਬੇ ਜਵਾਬਾਂ ਨੂੰ ਰੋਕਣ ਲਈ `max_tokens` ਦੀ ਸੀਮਾ ਨਿਰਧਾਰਤ ਕਰੋ। |
| **ਬੈਚਿੰਗ** | ਜਿੱਥੇ ਸੰਭਵ ਹੋਵੇ, ਕਈ ਯੂਜ਼ਰ ਪ੍ਰਸ਼ਨਾਂ ਨੂੰ ਇੱਕ ਏਪੀਆਈ ਕਾਲ ਵਿੱਚ ਸਮੂਹਿਤ ਕਰੋ। |

ਅਮਲੀ ਤੌਰ 'ਤੇ, ਇੱਕ ਪੜਾਅਵਾਰ ਨਜ਼ਰੀਆ ਚੰਗਾ ਕੰਮ ਕਰਦਾ ਹੈ: ਸਿੱਧੇ ਸਾਦੇ ਬੇਨਤੀਆਂ ਨੂੰ ਤੇਜ਼ ਅਤੇ ਸਸਤੇ ਮਾਡਲ ਵੱਲ ਰੂਟ ਕਰੋ ਅਤੇ ਸਿਰਫ਼ ਜਟਿਲ ਪ੍ਰਸ਼ਨਾਂ ਨੂੰ ਜ਼ਿਆਦਾ ਸਮਰੱਥ ਮਾਡਲ ਵੱਲ ਭੇਜੋ।


## ਸਰੰਸ਼

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ ਕਿ:

1. ਏਜੰਟ ਇੰਟਰੇਕਸ਼ਨਾਂ ਨਾਲ ਸਮੇਂ ਬੰਦ ਕਰਕੇ ਅਤੇ ਲਾਗਿੰਗ ਕਰਕੇ **ਨਾਂ ਭੂਲਣਯੋਗਤਾ ਸ਼ਾਮਲ ਕਰੋ**, ਜੋ ਟ੍ਰੇਸਿੰਗ ਅਤੇ ਮਾਨੀਟਰਿੰਗ ਲਈ ਮੂਲ ਤਿਆਰ ਕਰਦਾ ਹੈ।
2. ਇੱਕ ਮੁੱਲਾਂਕਣ ਏਜੰਟ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਏਜੰਟ ਜਵਾਬਾਂ ਨੂੰ **ਆਪਮੈਟਿਕ ਤੌਰ 'ਤੇ ਮੁੱਲਾਂਕਣ ਕਰੋ** ਜੋ ਪੂਰਨਤਾ, ਸਹੀਤਾ ਅਤੇ ਮਦਦਗਾਰਤਾ ਨੂੰ ਅੰਕਤ ਕਰਦਾ ਹੈ।
3. ਪ੍ਰੰਪਟ ਅਪਟੀਮਾਈਜੇਸ਼ਨ, ਮਾਡਲ ਚੋਣ, ਕੈਸ਼ਿੰਗ ਅਤੇ ਟੋਕਨ ਬਜਟ ਰਾਹੀਂ **ਲਾਗਤਾਂ ਦਾ ਪ੍ਰਬੰਧਨ ਕਰੋ**।

ਇਹ ਪ੍ਰੋਡਕਸ਼ਨ ਅਮਲ ਤੁਹਾਡੇ AI ਏਜੰਟਾਂ ਨੂੰ ਭਰੋਸੇਯੋਗ, ਮਾਪਣ ਯੋਗ ਅਤੇ ਪੈਸਾ-ਬਚਤ ਵਾਲੇ ਬਣਾਉਣ ਵਿੱਚ ਸਹਾਇਤਾ ਕਰਦੇ ਹਨ। 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
